In [0]:
from pyspark.sql.functions import expr, col, explode, monotonically_increasing_id, ai_parse_document, transform, array_join

In [0]:
dbutils.widgets.text("catalog_name", "rag_agentic")
dbutils.widgets.text("schema_name", "workday_demos")
dbutils.widgets.text("customer_docs", "customer_feedback")
dbutils.widgets.text("notes_docs", "meeting_notes")
dbutils.widgets.text("email_docs", "email_communications")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
customer_docs = dbutils.widgets.get("customer_docs")
notes_docs = dbutils.widgets.get("notes_docs")
email_docs = dbutils.widgets.get("email_docs")

In [0]:
def create_knowledge_base(catalog_name, schema_name, doc_name):
    # Get the max modification time
    max_modification_time = (spark.read.table(f"{catalog_name}.{schema_name}.{doc_name}_kb")
                                        .select("modification_time").agg({"modification_time": "max"})
                                        .first()[0] or '1900-01-01T00:00:00.000+00:00'
                            )

    # Read the customer feedback pdf files using binaryFile
    feedback_df = spark.read.format("binaryFile").load(f"/Volumes/{catalog_name}/{schema_name}/workday_unstructure_data/{doc_name}/").filter(col("modificationTime") > max_modification_time)

    # Parse the pdf files and extract the required information
    feedback_parsed = (feedback_df.withColumn("parsed", ai_parse_document("content", {"version": "2.0"}))
                                .withColumn("content", array_join(transform(expr("parsed:document.elements::ARRAY<STRUCT<content:STRING>>"), lambda x: x.content), "\n"))
                                .withColumn("document", expr("parsed:document"))
                                .withColumn("pages", expr("parsed:document:pages"))
                                .withColumn("error_status", expr("parsed:error_status"))
                                .withColumn("doc_uri", col("path"))
                                .select("content", "doc_uri", col("modificationTime").alias("modification_time"))
                                )

    #  Write the parsed document to the table
    feedback_parsed.write.mode("append").saveAsTable(f"{catalog_name}.{schema_name}.{doc_name}_kb")

In [0]:
# Customer feedback
create_knowledge_base(catalog_name, schema_name, customer_docs)

# notes feedback
create_knowledge_base(catalog_name, schema_name, notes_docs)

# email communications
create_knowledge_base(catalog_name, schema_name, email_docs)